# IDM-VTON — Gradio App (Public Link)

CatVTON kabi ishlatish — public URL olib brauzerda test qilish.

**Tartib:**
1. Cell 1 → run
2. Cell 2 → run
3. Cell 3 → run → **kernel o'zi restart bo'ladi** (kutiladi)
4. Restart bo'lgandan keyin: **faqat cell 4 va 5** ni run qiling

> Alohida Colab session'da oching (CatVTON Colab'dan mustaqil).

In [ ]:
# Cell 1 — GPU + Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import torch
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

In [ ]:
# Cell 2 — Clone
%cd /content
!rm -rf /content/IDM-VTON
!git clone https://github.com/yisol/IDM-VTON.git /content/IDM-VTON
%cd /content/IDM-VTON
print("Clone tayyor.")

In [ ]:
# Cell 3 — Dependencies
# Bu cell oxirida kernel avtomatik restart bo'ladi.
# Restart bo'lgandan keyin faqat cell 4 va 5 ni run qiling!

!pip install -q omegaconf einops accelerate

# huggingface_hub 0.23.4 — diffusers==0.25.0 bilan mos (yangi hub _get_model_file bilan muammoli)
!pip install -q "huggingface_hub==0.23.4"
!pip install -q "diffusers==0.25.0"

!pip install -q torchvision opencv-python Pillow
!pip install -q onnxruntime-gpu
!pip install -q "gradio==3.41.2" gradio-client
!pip install -q av  # densepose video moduli uchun

# detectron2: prebuilt wheel torch 2.10+cu128 bilan mos kelmaydi
# Source dan build (~10-15 daqiqa, faqat birinchi safar)
print("detectron2 build qilinmoqda (~10-15 daqiqa, kutib turing)...")
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

# numpy 1.x + scipy
!pip install -q "numpy>=1.24,<2.0" scipy

import numpy as np, torch
print(f"numpy: {np.__version__} | torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

try:
    import detectron2
    print(f"detectron2: {detectron2.__version__} OK")
except ImportError as e:
    print(f"detectron2 XATO (Cell 5 ishlashi mumkin emas): {e}")

import huggingface_hub as hfh
print(f"huggingface_hub: {hfh.__version__}")

print("\nKernel restart bo'ladi — keyin FAQAT cell 4 va 5 ni run qiling!")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Cell 4 — Modellarni Drive'ga yuklab, ckpt/ ga sozlash
import os, shutil, glob
from huggingface_hub import snapshot_download, hf_hub_download

CKP = "/content/IDM-VTON/ckpt"
os.makedirs(CKP, exist_ok=True)

def drive_symlink(drive_path, local_path):
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    if os.path.islink(local_path): os.unlink(local_path)
    if os.path.exists(local_path) and not os.path.islink(local_path): shutil.rmtree(local_path)
    os.symlink(drive_path, local_path)

# 1. IDM-VTON asosiy model
DRIVE_IDM = "/content/drive/MyDrive/Lookzi/hf_models/idm-vton"
if not os.path.exists(f"{DRIVE_IDM}/unet"):
    print("IDM-VTON yuklanmoqda (~8GB, 20-30 daqiqa)...")
    snapshot_download("yisol/IDM-VTON", local_dir=DRIVE_IDM)
else:
    print("IDM-VTON: Drive'da mavjud")
drive_symlink(DRIVE_IDM, f"{CKP}/IDM-VTON")

# 2. Humanparsing ONNX
# levihsu/GarmentDiffusion da fayllar ckpt/humanparsing/ subdirda joylashgan
DRIVE_HP = "/content/drive/MyDrive/Lookzi/hf_models/humanparsing"
os.makedirs(DRIVE_HP, exist_ok=True)

for fname in ["parsing_atr.onnx", "parsing_lip.onnx"]:
    fpath = f"{DRIVE_HP}/{fname}"
    if os.path.exists(fpath):
        print(f"{fname}: Drive'da mavjud")
        continue

    # Avval Drive'dagi subdirlardan qidirish (oldin noto'g'ri yuklanganlar)
    found_sub = glob.glob(f"{DRIVE_HP}/**/{fname}", recursive=True)
    if found_sub:
        shutil.copy(found_sub[0], fpath)
        print(f"{fname}: subdirdan ko'chirildi")
        continue

    # IDM-VTON snapshot ichidan qidirish
    found_idm = glob.glob(f"{DRIVE_IDM}/**/{fname}", recursive=True)
    if found_idm:
        shutil.copy(found_idm[0], fpath)
        print(f"{fname}: IDM-VTON repo'dan olindi")
        continue

    # HuggingFace'dan to'g'ridan-to'g'ri yuklab ko'chirish
    print(f"{fname} yuklanmoqda...")
    try:
        tmp_path = hf_hub_download(
            repo_id="levihsu/GarmentDiffusion",
            filename=f"ckpt/humanparsing/{fname}",
            local_dir="/tmp/hp_tmp"
        )
        shutil.copy(tmp_path, fpath)
        print(f"  {fname}: OK")
    except Exception as e:
        print(f"  XATO: {e}")
        print(f"  {fname} topilmadi — menga xabar bering")

drive_symlink(DRIVE_HP, f"{CKP}/humanparsing")

# 3. OpenPose
DRIVE_OP = "/content/drive/MyDrive/Lookzi/hf_models/openpose"
os.makedirs(DRIVE_OP, exist_ok=True)
op_pth = f"{DRIVE_OP}/body_pose_model.pth"
if not os.path.exists(op_pth):
    print("OpenPose yuklanmoqda...")
    try:
        p = hf_hub_download("lllyasviel/ControlNet",
                            "annotator/ckpts/body_pose_model.pth",
                            local_dir="/tmp/op")
        shutil.copy(p, op_pth)
        print("OpenPose: OK")
    except Exception as e:
        print(f"  XATO: {e}")
else:
    print("OpenPose: Drive'da mavjud")
os.makedirs(f"{CKP}/openpose/ckpts", exist_ok=True)
op_dst = f"{CKP}/openpose/ckpts/body_pose_model.pth"
if os.path.exists(op_pth) and not os.path.exists(op_dst):
    os.symlink(op_pth, op_dst)

# 4. Holat
print("\n--- Model holati ---")
for name, path in [
    ("IDM-VTON unet",    f"{DRIVE_IDM}/unet"),
    ("parsing_atr.onnx", f"{DRIVE_HP}/parsing_atr.onnx"),
    ("parsing_lip.onnx", f"{DRIVE_HP}/parsing_lip.onnx"),
    ("body_pose_model",  op_pth),
]:
    print(f"  {'OK' if os.path.exists(path) else 'YOQ'}: {name}")

In [ ]:
# Cell 5 — IDM-VTON Gradio App ishga tushirish
# Public link 'Running on public URL: https://...' qatorida chiqadi.

!fuser -k 7860/tcp 2>/dev/null; sleep 1

import sys, os, re, subprocess as _sp, glob as gl

sys.path.insert(0, '/content/IDM-VTON')
sys.path.insert(0, '/content/IDM-VTON/gradio_demo')  # utils_mask.py shu papkada

# diffusers o'rnatish joyini topish
def _diffusers_location():
    r = _sp.run(['pip', 'show', 'diffusers'], capture_output=True, text=True)
    for line in r.stdout.split('\n'):
        if line.startswith('Location:'):
            return line.split(':', 1)[1].strip()
    return None

_diff_loc = _diffusers_location()

# --- Patch 1: diffusers cached_download ---
_OLD = 'from huggingface_hub import cached_download, hf_hub_download, model_info'
_NEW = ('from huggingface_hub import hf_hub_download, model_info\n'
        'try:\n    from huggingface_hub import cached_download\n'
        'except ImportError:\n    cached_download = hf_hub_download')

if _diff_loc:
    _dmu = os.path.join(_diff_loc, 'diffusers', 'utils', 'dynamic_modules_utils.py')
    if os.path.exists(_dmu):
        _txt = open(_dmu).read()
        if _OLD in _txt:
            open(_dmu, 'w').write(_txt.replace(_OLD, _NEW))
            print("patch 1 (cached_download): OK")
        else:
            print("patch 1 (cached_download): allaqachon qilingan")

# --- Patch 2: transformers FLAX_WEIGHTS_NAME ---
import transformers.utils as _tu
_missing = {'FLAX_WEIGHTS_NAME': 'flax_model.msgpack',
            'TF2_WEIGHTS_NAME': 'tf_model.h5', 'TF_WEIGHTS_NAME': 'model.ckpt'}
_added = [k for k, v in _missing.items() if not hasattr(_tu, k) and not setattr(_tu, k, v)]
print(f"patch 2 (transformers): {', '.join(_added) + ' qo\\'shildi' if _added else 'shart emas'}")

# --- Patch 3: huggingface_hub 1.x da proxies/resume_download deprecated ---
# Validator ularni strips qiladi, lekin inner func hali ham required deb kutadi.
# Yechim: diffusers fayllaridan bu deprecated arglarni olib tashlash.
if _diff_loc:
    _patched3 = []
    for _py in gl.glob(f"{_diff_loc}/diffusers/**/*.py", recursive=True):
        try:
            _t = open(_py).read()
            if '_get_model_file' not in _t and 'from_pretrained' not in _t:
                continue
            _orig = _t
            # resume_download=resume_download va proxies=proxies qatorlarini olib tashlash
            _t = re.sub(r'\n[ \t]+resume_download\s*=\s*resume_download\s*,', '', _t)
            _t = re.sub(r'\n[ \t]+proxies\s*=\s*proxies\s*,', '', _t)
            if _t != _orig:
                open(_py, 'w').write(_t)
                _patched3.append(os.path.basename(_py))
        except Exception:
            pass
    if _patched3:
        print(f"patch 3 (proxies/resume_download): {', '.join(_patched3)}")
    else:
        print("patch 3 (proxies/resume_download): shart emas")

# --- IDM-VTON launch ---
try:
    import gradio_demo.app as idm_app
    demo = getattr(idm_app, 'demo', None)
    if demo is None:
        raise AttributeError("'demo' topilmadi — app.py tuzilishini tekshiring")
    print("\nGradio app ishga tushmoqda...")
    demo.queue().launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=True,
        show_error=True,
    )
except Exception as e:
    import traceback
    print(f"\nXATO: {e}")
    traceback.print_exc()
    print("\n--- Xato matnini menga yuboring ---")